# 🟠 Day 9 — Streaming & Async

**LangGraph from Zero to Advanced · Day 9 of 16**

**Today's goal:** show users what's happening **live** — tokens as they are typed, progress messages, tool calls — and run graphs **async** for speed.

| Lesson | Topic |
|---|---|
| 1 | Stream modes at a glance |
| 2 | Stream LLM tokens (`"messages"`) |
| 3 | Only stream tokens from one node |
| 4 | Send your own progress events (`"custom"`) |
| 5 | Several stream modes at once |
| 6 | Watch an agent's tool calls live |
| 7 | Async: `ainvoke` / `astream` and parallel speed-up |
| 8 | Practice |

In [ ]:
%pip install -q -U langgraph langchain-openai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from IPython.display import Image, display

load_dotenv(find_dotenv(usecwd=True))
assert os.getenv("OPENAI_API_KEY"), "Add OPENAI_API_KEY to the .env file in the repo root (see Day 4)."
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-nano"), temperature=0)

def show_graph(app):
    """Draw the graph as a picture, or print it as text when offline."""
    try:
        display(Image(app.get_graph().draw_mermaid_png()))
    except Exception:
        print(app.get_graph().draw_mermaid())

print("✅ Ready")

## Lesson 1 — Stream modes at a glance

| `stream_mode` | What you get | Use it for |
|---|---|---|
| `"values"` | full state after each step | debugging |
| `"updates"` | `{node: what it returned}` after each step | progress ("step X done") |
| `"messages"` | LLM **tokens** as `(chunk, metadata)` | chat UIs (typing effect) |
| `"custom"` | anything you send with `get_stream_writer()` | your own progress events |
| `"debug"` | everything, very detailed | deep debugging |

We'll use a small 2-node graph: write a story outline, then write the story.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class StoryState(TypedDict):
    topic: str
    outline: str
    story: str

def outline(state):
    return {"outline": llm.invoke(f"Write a 3-point outline for a tiny story about {state['topic']}.").content}

def write_story(state):
    return {"story": llm.invoke(f"Write a 5-sentence story from this outline:\n{state['outline']}").content}

g = StateGraph(StoryState)
g.add_sequence([outline, write_story])
g.add_edge(START, "outline")
g.add_edge("write_story", END)
story_app = g.compile()

for chunk in story_app.stream({"topic": "a robot learning to cook biryani"}, stream_mode="updates"):
    node = list(chunk)[0]
    print(f"✅ {node} finished")

## Lesson 2 — Stream LLM tokens

With `stream_mode="messages"`, LangGraph captures every **token** from any LLM call **inside any node** — even though the node uses `llm.invoke()`! Each item is `(token_chunk, metadata)`.

In [ ]:
for token, metadata in story_app.stream({"topic": "a lazy cat"}, stream_mode="messages"):
    print(token.content, end="", flush=True)
print()

## Lesson 3 — Only stream tokens from one node

`metadata["langgraph_node"]` tells you which node produced the token. Show only the final story, not the outline:

In [ ]:
for token, metadata in story_app.stream({"topic": "a brave little auto-rickshaw"}, stream_mode="messages"):
    if metadata["langgraph_node"] == "write_story":
        print(token.content, end="", flush=True)
print()

## Lesson 4 — Send your own progress events

Inside a node, `get_stream_writer()` gives you a `writer` function. Anything you pass to it is streamed with `stream_mode="custom"`. Great for "🔍 searching… 📄 reading 3 files…" messages.

In [ ]:
import time
from langgraph.config import get_stream_writer

class ReportState(TypedDict):
    files: list[str]
    report: str

def process_files(state):
    writer = get_stream_writer()
    for i, f in enumerate(state["files"], 1):
        writer({"progress": f"📄 reading {f} ({i}/{len(state['files'])})"})
        time.sleep(0.3)                       # pretend work
    writer({"progress": "✍️ writing report"})
    return {"report": f"Processed {len(state['files'])} files"}

g = StateGraph(ReportState)
g.add_node("process_files", process_files)
g.add_edge(START, "process_files")
g.add_edge("process_files", END)
report_app = g.compile()

for event in report_app.stream({"files": ["sales.csv", "users.csv", "orders.csv"]}, stream_mode="custom"):
    print(event["progress"])

## Lesson 5 — Several stream modes at once

Pass a **list** of modes. Each item becomes a tuple `(mode, data)`.

In [ ]:
for mode, data in report_app.stream({"files": ["a.txt", "b.txt"]}, stream_mode=["custom", "updates"]):
    print(f"[{mode:7}]", data)

## Lesson 6 — Watch an agent's tool calls live

Combine `"updates"` with the agent from Day 6 to show each tool call as it happens.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def get_stock_price(symbol: str) -> str:
    """Get the latest stock price for an NSE symbol (demo data)."""
    prices = {"TCS": 4120.5, "INFY": 1890.0, "RELIANCE": 2950.25}
    return f"{symbol.upper()}: ₹{prices.get(symbol.upper(), 'unknown')}"

agent = create_agent(model=llm, tools=[get_stock_price],
                     system_prompt="Answer stock questions using the tool.")

for chunk in agent.stream({"messages": [("user", "Compare the prices of TCS and INFY")]}, stream_mode="updates"):
    for node, update in chunk.items():
        for msg in update.get("messages", []):
            if getattr(msg, "tool_calls", None):
                for c in msg.tool_calls:
                    print(f"🔧 calling {c['name']}({c['args']})")
            elif msg.type == "tool":
                print(f"📦 result: {msg.content}")
            else:
                print(f"🤖 {msg.content}")

## Lesson 7 — Async: `ainvoke` / `astream`

Web apps (FastAPI etc.) are async. Every method has an async twin: `ainvoke`, `astream`, `llm.ainvoke`. In Jupyter you can use `await` directly in a cell.

Async shines when **parallel** nodes wait on the network. Three LLM calls in parallel take about as long as one.

In [ ]:
import asyncio, operator
from typing import Annotated

class FactsState(TypedDict):
    facts: Annotated[list[str], operator.add]

def make_fact_node(topic):
    async def node(state):                                   # 👈 async node
        msg = await llm.ainvoke(f"One short fun fact about {topic}.")
        return {"facts": [f"{topic}: {msg.content}"]}
    return node

g = StateGraph(FactsState)
for topic in ("space", "oceans", "honeybees"):
    g.add_node(topic, make_fact_node(topic))
    g.add_edge(START, topic)       # all 3 run in parallel
    g.add_edge(topic, END)
facts_app = g.compile()
show_graph(facts_app)

start = time.time()
result = await facts_app.ainvoke({"facts": []})
print(f"⏱ {time.time() - start:.1f}s for 3 LLM calls in parallel\n")
for f in result["facts"]:
    print("•", f)

In [ ]:
# astream works the same way — use `async for`
async for token, meta in story_app.astream({"topic": "a monsoon day"}, stream_mode="messages"):
    if meta["langgraph_node"] == "write_story":
        print(token.content, end="", flush=True)
print()

💡 In a normal `.py` script (not Jupyter), wrap async code like this: `asyncio.run(main())`.

## Lesson 8 — Practice

**Exercise 1.** Stream `story_app` with `stream_mode="values"` and print only how many keys the state has after each step.

**Exercise 2.** Write a node that "downloads" 5 pages and sends a custom event with a **percentage** (20%, 40% …). Print a simple text progress bar like `[####------] 40%`.

In [ ]:
# ✅ Solution 1
for state in story_app.stream({"topic": "a kite"}, stream_mode="values"):
    print("keys in state:", len(state), list(state))

In [ ]:
# ✅ Solution 2
class DownloadState(TypedDict):
    done: bool

def download(state):
    writer = get_stream_writer()
    for page in range(1, 6):
        time.sleep(0.2)
        writer({"pct": page * 20})
    return {"done": True}

g = StateGraph(DownloadState)
g.add_node("download", download)
g.add_edge(START, "download")
g.add_edge("download", END)
for ev in g.compile().stream({}, stream_mode="custom"):
    bars = ev["pct"] // 10
    print(f"[{'#' * bars}{'-' * (10 - bars)}] {ev['pct']}%")
print("✅ All exercises solved!")

## 🎯 Day 9 Recap

* `"updates"` → step progress, `"messages"` → LLM tokens, `"custom"` → your events, `"values"` → full state.
* `metadata["langgraph_node"]` filters tokens by node.
* `get_stream_writer()` sends custom events from inside nodes.
* Async twins: `ainvoke`, `astream`, `async def` nodes → parallel calls run concurrently.

🎉 **You've finished the core course (Days 1–9).** From Day 10 we build advanced patterns.

**Tomorrow (Day 10):** structured output + the 5 classic **workflow patterns**.